# AirfRANS GNN Surrogate — Kaggle GPU setup

Fallback for when Colab's free GPU quota runs out -- separate quota pool
(~30 hrs/week of P100 or T4x2). No Drive-equivalent live mount here: Kaggle
persists across sessions via **Datasets** (read-only, attached to a session)
and a notebook's own **Output** (from "Save Version"), not a synced folder.

Before running: Settings (right sidebar) > Accelerator > GPU, and
Internet > On (needed for git clone / pip install / dataset download).

## 0. This run starts fresh, deliberately -- not resuming

Earlier weighted-loss run (epoch 9 -> ... -> epoch 91, checkpoints in
`meshgraphnet_weighted.ckpt`'s directory) confirmed a real, matched-sample
regression: true Cd relative L2 was best at epoch 47 (3.15) and *worse* at
the final epoch 91 (3.97), despite every field-level metric and Cl improving
the whole way through (see `ARCHITECTURE.md` sections 10-11). Splitting Cd
into pressure drag (`cdp`) and friction drag (`cdv`) found the failure mode
isn't fixed across training, it *moves* -- and a constant-prediction
baseline (35.1% relative L2) beat both checkpoints outright.

This run's change: `src/graph.py` now includes `simulation.normals` (a unit
vector at surface nodes, zero elsewhere) as two extra node features
(`node_in_dim` 5 -> 7) -- `wall_distance` alone told the model how far a
node was from the wall but never which direction was "into" it, so it had
no way to represent the boundary layer's anisotropy from a single scalar.
This changes the node encoder's input shape, so **old checkpoints (any of
them) cannot be resumed from** -- a shape-mismatched `load_state_dict` would
error immediately, not silently corrupt anything, but a new checkpoint path
avoids the error entirely and keeps this run's results unambiguous.

**Before running this notebook: push the local `src/graph.py`, `src/model.py`,
`src/dataset.py`, and `data/norm_stats.npz` changes to GitHub.** Cell 5
below does a fresh `git clone` from the repo URL -- Kaggle has no access to
this machine's working directory, so if those changes aren't pushed, this
notebook trains the *old* 5-feature model under a new name, silently.

**Only the normals feature is being changed in this run, deliberately.** A
wall-shear-gradient auxiliary loss was also drafted (`src/train.py`'s
`wall_shear_gradient_proxy`) but found not reliable on this dataset's actual
mesh topology during local testing (see `ARCHITECTURE.md` section 11) --
its weight defaults to 0.0 and the training call below doesn't override it.
Changing two things in one Kaggle run would confound the results if Cd is
still bad afterward: normals alone first, so a regression (or improvement)
can be attributed cleanly.

In [ ]:
# New, distinct checkpoint path -- keeps this run's checkpoints separate from
# both the old unweighted-loss run and the weighted-loss run (5 node
# features), so auto-resume in the training cell below finds nothing here
# and starts fresh at epoch 0, deliberately.
CHECKPOINT_PATH = "/kaggle/working/meshgraphnet_normals.ckpt"

import glob
import os

existing = glob.glob(os.path.join(os.path.dirname(CHECKPOINT_PATH), "mgn-epoch=*.ckpt"))
print("existing checkpoints at this path (should be empty for a fresh start):", existing)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the repo and install dependencies

Same as Colab -- Kaggle also ships CUDA-enabled torch preinstalled, and
`torch_geometric` installs as pure Python (no `torch-scatter`/`torch-sparse`
needed, confirmed working on both machines already).

In [ ]:
REPO_URL = "https://github.com/Revanthkr1/airfrans-gnn-surrogate.git"

!git clone $REPO_URL repo
%cd repo
!pip install -q torch_geometric lightning airfrans pyvista

## 2. Download + preprocess, one case at a time

`af.dataset.download(unzip=True)` needs the zip (~9.34GB) *and* the fully
extracted dataset (~15GB) on disk simultaneously to extract everything at
once -- that alone exceeded this session's disk quota (`OSError: No space
left on device`, mid-extraction, before preprocessing even started).

Instead: download just the zip, then extract + cache one case at a time,
deleting each case's raw files immediately after caching. Peak disk usage
stays at roughly (zip + one case + the cache built so far) instead of
(zip + the entire raw dataset) at once. `manifest.json` doesn't need
extracting from the zip either -- it's committed to the repo at
`data/manifest.json`.

In [ ]:
from src.data import split_names
from src.preprocess import download_zip_only, stream_preprocess_from_zip

DATA_ROOT = "data"
WORK_DIR = "/kaggle/working/raw"  # transient extraction scratch, one case at a time
CACHE_DIR = "/kaggle/working/cache/full"
MANIFEST_DIR = "data"  # data/manifest.json is committed to the repo -- always present

train_names = split_names(MANIFEST_DIR, task="full", train=True)
zip_path = download_zip_only(DATA_ROOT)
stream_preprocess_from_zip(zip_path, train_names, CACHE_DIR, WORK_DIR)

import os
os.remove(zip_path)  # done with it -- frees ~9.34GB back
print(f"cached {len(os.listdir(CACHE_DIR))}/{len(train_names)} training cases")

## 3. Train, fresh, with the normals node feature

No `resume_from_checkpoint` passed -- `CHECKPOINT_PATH` (section 0) is a new,
empty directory, so auto-detect finds nothing and this starts at epoch 0 on
purpose. Everything else is unchanged from the weighted-loss run:
`batch_size=1` + `accumulate_grad_batches=4` (OOM headroom),
`precision="16-mixed"`, `checkpoint_every_n_epochs=5`, and the same
distance-weighted loss (`wall_weight_peak`/`wall_weight_length_scale`
defaults). `model_kwargs=None` below means `MeshGraphNet`'s own default
`node_in_dim=7` is picked up automatically -- no explicit override needed,
since `DEFAULT_MODEL_KWARGS` (`src/train.py`) only sets
latent/hidden/message-passing sizes, not `node_in_dim`.

Once this finishes (or periodically during it), use
`notebooks/compare_checkpoints.py` locally to check whether the normals
feature actually helps -- against the epoch-47 and epoch-91 weighted-loss
checkpoints as the baseline to beat, the same way the epoch-54-vs-99 and
epoch-47-vs-91 regressions were originally found. Don't just trust the
final epoch, and don't trust a small subset either -- see `ARCHITECTURE.md`
section 10 on why a matched, full-split comparison mattered there.

**To persist past this session**: click "Save Version" when done (or
periodically) -- that's Kaggle's equivalent of Drive surviving a disconnect.

In [ ]:
from src.train import main as train_main

train_main(
    dataset_root=MANIFEST_DIR,
    cache_dir=CACHE_DIR,
    stats_path="data/norm_stats.npz",
    checkpoint_path=CHECKPOINT_PATH,
    max_epochs=100,
    batch_size=1,
    accumulate_grad_batches=4,
    n_val=80,
    checkpoint_every_n_epochs=5,
    num_workers=2,
    precision="16-mixed",
)